# CarHE Tutorial: From Data to Training to Inference

This notebook walks through the complete CarHE pipeline:
1. **Data preparation** — Convert Xenium/Visium data to AnnData (.h5ad)
2. **Training** — Train the HIPT + CLIP model
3. **Evaluation** — Assess model performance
4. **Inference** — Predict gene expression on new images
5. **GradCAM** — Visualize tissue regions driving predictions

> **Prerequisites:** Installed dependencies (see README.md), downloaded HIPT weights, and prepared data.

## 0. Setup & Imports

In [ ]:
import os
import sys
import torch
import numpy as np
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
from tqdm import tqdm

# Add model directory to path
sys.path.insert(0, './model')

from config import CFG
from model import HIPT_CLIP_Model, ProjectionHead
from get_adata import SpotDataset, build_loaders_adata
from gradcam import GradCAM_CarHE, visualize_heatmap
from utils import AvgMeter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

## 1. Data Preparation

CarHE requires data in **AnnData (.h5ad)** format. Convert your data using the built-in converters.

### Option A: Xenium data (built-in quick converter)

In [ ]:
# Convert Xenium raw output to h5ad
# Run this in the terminal (requires reading raw cell_feature_matrix.h5):
# cd data
# python quick_xenium_to_h5ad.py

# Load the converted data
adata_path = '../data/xenium_prostate.h5ad'

if os.path.exists(adata_path):
    adata = sc.read_h5ad(adata_path)
    print(f'Data loaded: {adata.shape}')
    print(f'Samples: {adata.obs["sample_id"].unique().tolist()}')
    print(f'Genes: {adata.n_vars}')
    print(f'First 5 genes: {adata.var_names[:5].tolist()}')
else:
    print(f'Run quick_xenium_to_h5ad.py first to generate {adata_path}')

### Option B: Manual AnnData creation (for custom data)

Build an AnnData from scratch if you have your own images and expression data.

In [ ]:
# Example: Build AnnData from custom data

# import pandas as pd
# 
# # Load your data
# expression = pd.read_csv('my_expression.csv', index_col=0)      # cells x genes
# coordinates = pd.read_csv('my_coordinates.csv')                # cell_id, x, y
# image_path = '/absolute/path/to/my_he_image.tif'
# 
# # Create AnnData
# adata = ad.AnnData(
#     X=expression.values.astype(np.float32),
#     obs=pd.DataFrame({
#         'sample_id': ['my_sample'] * len(expression),
#         'image_path': [image_path] * len(expression),
#     }, index=expression.index),
#     var=pd.DataFrame(index=expression.columns),
# )
# adata.obsm['spatial'] = coordinates[['x', 'y']].values.astype(np.float32)
# adata.write_h5ad('my_data.h5ad')
# print(f'Saved: my_data.h5ad ({adata.shape})')

print('See code comments above for manual AnnData creation.')

### Validate AnnData format

In [ ]:
def validate_adata(adata):
    """Check if AnnData meets CarHE requirements"""
    checks = {
        '.X exists': adata.X is not None,
        "obs['sample_id']": 'sample_id' in adata.obs,
        "obs['image_path']": 'image_path' in adata.obs,
        "obsm['spatial']": 'spatial' in adata.obsm,
    }
    for name, ok in checks.items():
        print(f"  {'✓' if ok else '✗'} {name}")
    
    if 'spatial' in adata.obsm:
        sp = adata.obsm['spatial']
        print(f"  Spatial range: x=[{sp[:,0].min():.0f}, {sp[:,0].max():.0f}], "
              f"y=[{sp[:,1].min():.0f}, {sp[:,1].max():.0f}]")
    
    print(f"\n  Shape: {adata.shape}")
    print(f"  Memory: {adata.X.nbytes / 1024**2:.1f} MB")

if 'adata' in dir():
    validate_adata(adata)
else:
    print('No adata loaded yet. Run Option A or B first.')

## 2. Build Data Loaders

CarHE uses a unified data loader that reads AnnData and creates PyTorch DataLoaders.

In [ ]:
# Build train/validation loaders
if 'adata' in dir():
    train_loader, test_loader = build_loaders_adata(
        adata=adata,
        batch_size=32,
        num_workers=4,
        train_ratio=0.8,
    )
    
    # Inspect a batch
    batch = next(iter(train_loader))
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            print(f'{k}: shape={list(v.shape)}')
        else:
            print(f'{k}: {v[0] if isinstance(v, list) else v}')
else:
    print('No adata loaded yet.')

In [ ]:
# Visualize sample image patches
if 'batch' in dir():
    fig, axes = plt.subplots(1, 4, figsize=(12, 3))
    for i in range(4):
        img = batch['image'][i].permute(1, 2, 0).cpu().numpy()
        img = (img - img.min()) / (img.max() - img.min())
        axes[i].imshow(img)
        axes[i].set_title(f'Cell {i}')
        axes[i].axis('off')
    plt.suptitle('Sample H&E Patches (256x256)', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No batch loaded yet.')

## 3. Model Initialization

CarHE uses HIPT ViT-256 as the image encoder and a CLIP-style contrastive learning framework.

In [ ]:
# Initialize the model
model = HIPT_CLIP_Model().to(device)
model.train()

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:      {total_params:>12,}')
print(f'Trainable parameters:  {trainable_params:>12,}')
print(f'Trainable ratio:       {trainable_params/total_params*100:.1f}%')

# Show model components
print('\nModel components:')
for name, module in model.named_children():
    print(f'  {name}: {module.__class__.__name__}')

## 4. Training Loop

Train the model with the CLIP contrastive loss.

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    """Train for one epoch"""
    model.train()
    loss_meter = AvgMeter()
    
    for batch in tqdm(loader, desc='Training'):
        batch = {k: v.to(device) if k != 'barcode' else v for k, v in batch.items()}
        
        loss = model(batch)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        loss_meter.update(loss.item(), batch['image'].size(0))
    
    return loss_meter.avg


def validate_one_epoch(model, loader, device):
    """Validate for one epoch"""
    model.eval()
    loss_meter = AvgMeter()
    
    with torch.no_grad():
        for batch in tqdm(loader, desc='Validating'):
            batch = {k: v.to(device) if k != 'barcode' else v for k, v in batch.items()}
            loss = model(batch)
            loss_meter.update(loss.item(), batch['image'].size(0))
    
    return loss_meter.avg


# Run training
if 'train_loader' in dir() and 'test_loader' in dir():
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.002)
    
    best_loss = float('inf')
    history = {'train_loss': [], 'val_loss': []}
    
    for epoch in range(5):  # Train for 5 epochs as a demo
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_loss = validate_one_epoch(model, test_loader, device)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        
        print(f'Epoch {epoch+1:2d} | Train: {train_loss:.4f} | Val: {val_loss:.4f}')
        
        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), './checkpoint/demo_model.pt')
            print(f'  -> Saved best model (val_loss={best_loss:.4f})')
    
    # Plot loss curve
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history['train_loss'], label='Train Loss', marker='o')
    ax.plot(history['val_loss'], label='Val Loss', marker='s')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title('Training Progress'); ax.legend()
    plt.show()
else:
    print('No data loaders. Run Section 2 first.')

## 5. Embedding Extraction

Extract image and gene expression embeddings from the trained model.

In [ ]:
def extract_embeddings(model, loader, device):
    """Extract image and spot embeddings from a data loader"""
    model.eval()
    img_embs, spot_embs, barcodes = [], [], []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc='Extracting embeddings'):
            images = batch['image'].to(device)
            spots = batch['reduced_expression'].to(device)
            
            img_emb = model.encode_image(images)
            spot_emb = model.encode_spot(spots)
            
            img_embs.append(img_emb.cpu().numpy())
            spot_embs.append(spot_emb.cpu().numpy())
            barcodes.extend(batch['barcode'])
    
    return np.concatenate(img_embs), np.concatenate(spot_embs), barcodes


if 'test_loader' in dir() and os.path.exists('./checkpoint/demo_model.pt'):
    state = torch.load('./checkpoint/demo_model.pt', map_location=device)
    model.load_state_dict(state)
    
    img_embs, spot_embs, barcodes = extract_embeddings(model, test_loader, device)
    print(f'Image embeddings:     {img_embs.shape}')
    print(f'Expression embeddings: {spot_embs.shape}')
    
    # Compute cosine similarity
    from sklearn.preprocessing import normalize
    img_norm = normalize(img_embs)
    spot_norm = normalize(spot_embs)
    cos_sim = spot_norm @ img_norm.T
    diag = np.diag(cos_sim)
    off_diag = cos_sim[~np.eye(len(cos_sim), dtype=bool)]
    
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(diag, bins=50, alpha=0.6, label=f'Matched pairs (mean={diag.mean():.3f})')
    ax.hist(off_diag, bins=50, alpha=0.6, label=f'Random pairs (mean={off_diag.mean():.3f})')
    ax.set_xlabel('Cosine Similarity'); ax.set_ylabel('Frequency')
    ax.legend(); ax.set_title('Embedding Space Alignment')
    plt.show()
else:
    print('Need trained model (Section 4) to extract embeddings.')

## 6. GradCAM Visualization

Visualize which tissue regions the model focuses on for gene expression prediction.

In [ ]:
# GradCAM on a single cell
if 'model' in dir() and 'test_loader' in dir():
    model.eval()
    
    # Create GradCAM
    gradcam = GradCAM_CarHE(model, target_block_index=-1)
    
    # Get a single sample
    batch = next(iter(test_loader))
    image = batch['image'][0:1].to(device)
    expression = batch['reduced_expression'][0:1].to(device)
    barcode = batch['barcode'][0]
    
    # Compute heatmap
    heatmap, score = gradcam.compute(image, expression)
    print(f'GradCAM score for {barcode}: {score:.4f}')
    
    # Visualize
    orig_img = image.squeeze(0).permute(1, 2, 0).cpu().numpy()
    orig_img = ((orig_img - orig_img.min()) / (orig_img.max() - orig_img.min()) * 255).astype(np.uint8)
    
    fig = visualize_heatmap(heatmap, orig_img, title=f'GradCAM - {barcode}', score=score)
    plt.show()
else:
    print('Need model and data loader.')

## 7. KNN Gene Expression Prediction

Predict gene expression for new image patches using KNN in the embedding space.

In [ ]:
def knn_predict(query_embeddings, ref_embeddings, ref_expression, top_k=5):
    """Predict gene expression via KNN"""
    distances = torch.cdist(
        torch.tensor(query_embeddings),
        torch.tensor(ref_embeddings)
    )
    _, indices = torch.topk(distances, k=min(top_k, len(ref_embeddings)), 
                            dim=1, largest=False)
    
    predicted = np.zeros((len(query_embeddings), ref_expression.shape[1]))
    for i in range(len(query_embeddings)):
        predicted[i] = ref_expression[indices[i].numpy()].mean(axis=0)
    
    return predicted


if 'img_embs' in dir() and 'spot_embs' in dir() and 'adata' in dir():
    # Use first half as reference, second half as query
    n = len(img_embs)
    ref_img = img_embs[:n//2]
    ref_expr = adata.X[:n//2].toarray() if hasattr(adata.X, 'toarray') else adata.X[:n//2]
    query_img = img_embs[n//2:n]
    true_expr = adata.X[n//2:n].toarray() if hasattr(adata.X, 'toarray') else adata.X[n//2:n]
    
    predicted = knn_predict(query_img, ref_img, ref_expr, top_k=5)
    
    # Compute per-gene PCC
    from scipy.stats import pearsonr
    pccs = []
    for g in range(min(10, predicted.shape[1])):  # First 10 genes
        if np.std(predicted[:, g]) > 1e-8 and np.std(true_expr[:, g]) > 1e-8:
            r, _ = pearsonr(predicted[:, g], true_expr[:, g])
            pccs.append(r)
    
    print(f'Mean PCC (top 10 genes): {np.mean(pccs):.4f}')
    
    # Scatter plot
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for i, g in enumerate([0, 1, 2]):
        axes[i].scatter(true_expr[:, g], predicted[:, g], alpha=0.3, s=2)
        axes[i].set_xlabel('True Expression'); axes[i].set_ylabel('Predicted')
        axes[i].set_title(f'Gene {g}')
    plt.suptitle('Predicted vs True Gene Expression', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('Need embeddings from Section 5.')

## 8. Command-Line Reference

All steps can also be run from the command line:

In [ ]:
print('''
# === Data Conversion ===
cd data
python quick_xenium_to_h5ad.py                          # Xenium → h5ad
python convert_to_h5ad.py visium --spaceranger_dir ./   # Visium → h5ad
python convert_to_h5ad.py validate --adata data.h5ad    # Validate format

# === Training ===
cd ../model
python train.py --adata ../data/xenium_prostate.h5ad \\
    --batch_size 32 --epochs 50 --exp_name ./checkpoint/my_model

# === Evaluation ===
python evaluate.py --adata ../data/xenium_prostate.h5ad \\
    --checkpoint ./checkpoint/my_model/best_model.pt

# === GradCAM ===
python run_gradcam.py --adata ../data/xenium_prostate.h5ad \\
    --checkpoint ./checkpoint/my_model/best_model.pt --spot_idx 0

# === One-click Pipeline ===
cd ..
python run_pipeline.py --check                            # Check environment
python run_pipeline.py --all --epochs 50                  # Run everything
''')